In [42]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pickle

In [43]:
##Loading the dataset
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [44]:
##Preprocessing the data
##Dropping unnecessary columns
data.columns  # inspect column names

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='object')

In [45]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, errors='ignore')
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [46]:
##geography and Gender are categorical variables, we need to encode them
le = LabelEncoder()
data['Gender'] = le.fit_transform(data['Gender'])
data ##gender is now encoded as 0 and 1, where 0 represents 'Female' and 1 represents 'Male'

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [47]:
##geography is also a categorical variable, we need to encode it using one-hot encoding as it has more than two categories 
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder()  
geography_encoded = ohe.fit_transform(data[['Geography']])
geography_encoded

<10000x3 sparse matrix of type '<class 'numpy.float64'>'
	with 10000 stored elements in Compressed Sparse Row format>

In [48]:
ohe.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [49]:
geography_encoded_df = pd.DataFrame(geography_encoded.toarray(), columns=ohe.get_feature_names_out(['Geography']))
geography_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [50]:
##Combining the encoded geography columns with the original dataset and dropping the original Geography column
data = pd.concat([data, geography_encoded_df], axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,France,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,France,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [51]:
##Droping the original Geography column
data = pd.concat([data.drop('Geography', axis=1), geography_encoded_df], axis=1)
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0,0.0,1.0,0.0


In [52]:
## Save the encoders and the scaler for future use in pickle files
with open('label_encoder.pkl', 'wb') as le_file:
    pickle.dump(le, le_file)
with open('onehot_encoder.pkl', 'wb') as ohe_file:
    pickle.dump(ohe, ohe_file)

In [53]:
## Divide the dataset into independent and dependent features
X = data.drop('Exited', axis=1)  # Independent features
y = data['Exited']  # Dependent feature

##Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

##Scaling the features using StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [54]:
with open('scaler.pkl', 'wb') as scaler_file:
    pickle.dump(scaler, scaler_file)

In [55]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0,0.0,1.0,0.0


## ANN Implemantation

In [56]:
!pip uninstall tensorflow -y
!pip install tensorflow==2.10.1

Found existing installation: tensorflow 2.10.1
Uninstalling tensorflow-2.10.1:


ERROR: Exception:
Traceback (most recent call last):
  File "c:\users\shash\appdata\local\programs\python\python38\lib\shutil.py", line 791, in move
    os.rename(src, real_dst)
PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\users\\shash\\appdata\\local\\programs\\python\\python38\\scripts\\tensorboard.exe' -> 'C:\\Users\\shash\\AppData\\Local\\Temp\\pip-uninstall-rmyaxigr\\tensorboard.exe'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\users\shash\appdata\local\programs\python\python38\lib\site-packages\pip\_internal\cli\base_command.py", line 106, in _run_wrapper
    status = _inner_run()
  File "c:\users\shash\appdata\local\programs\python\python38\lib\site-packages\pip\_internal\cli\base_command.py", line 97, in _inner_run
    return self.run(options, args)
  File "c:\users\shash\appdata\local\programs\python\python38\lib\site-packages\pip\_inter

  Using cached tensorflow-2.10.1-cp38-cp38-win_amd64.whl.metadata (3.1 kB)
Using cached tensorflow-2.10.1-cp38-cp38-win_amd64.whl (455.9 MB)


ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\users\\shash\\appdata\\local\\programs\\python\\python38\\Scripts\\tensorboard.exe' -> 'c:\\users\\shash\\appdata\\local\\programs\\python\\python38\\Scripts\\tensorboard.exe.deleteme'
Consider using the `--user` option or check the permissions.



In [57]:
import tensorflow as tf
from tensorflow.keras.layers import Dense 
##Importing the Dense layer for building the neuron in hidden layers and output layer
from tensorflow.keras.models import Sequential 
##Importing the Sequential model for building the neural network
from tensorflow.keras.callbacks import EarlyStopping , TensorBoard 
##Importing EarlyStopping for preventing overfitting and TensorBoard for visualizing the training process
import datetime

In [58]:
X_train.shape ##Checking the shape of the training data to determine the input dimension for the neural network

(8000, 15)

In [59]:
##Building the neural network model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),  # Hidden layer 1 with 64 neurons and ReLU activation 
    Dense(32, activation='relu'),  # Hidden layer 2 with 32 neurons and ReLU activation
    Dense(1, activation='sigmoid')  # Output layer with 1 neuron and sigmoid activation for binary classification
]
)

In [60]:
model.summary() ##Summary of the model architecture

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                1024      
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 
Total params: 3,137
Trainable params: 3,137
Non-trainable params: 0
_________________________________________________________________


In [61]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TensorFlow warnings

In [62]:
import tensorflow
opt  = tensorflow.keras.optimizers.Adam(learning_rate=0.01) ##Defining the optimizer with a learning rate of 0.001
loss = tensorflow.keras.losses.BinaryCrossentropy() ##Defining the loss function for binary classification

In [63]:
##Compiling the model with binary crossentropy loss function and Adam optimizer
model.compile(optimizer=opt, loss=loss, metrics=['accuracy'])

In [64]:
##Setup TensorBoard callback for visualizing the training process
from tensorflow.keras.callbacks import TensorBoard

log_dir = "logs/fit" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)


In [65]:
##Setting up EarlyStopping and TensorBoard callbacks
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [66]:
##Training the model with the training data and validating on the test data, using the defined callbacks
history = model.fit(
    X_train, y_train, validation_data=(X_test, y_test), epochs=100,
    callbacks=[early_stopping_callback, tensorflow_callback]
)

Epoch 1/100
250/250 [==============================] - 1s 3ms/step - loss: 0.4032 - accuracy: 0.8319 - val_loss: 0.3643 - val_accuracy: 0.8555
Epoch 2/100
250/250 [==============================] - 1s 2ms/step - loss: 0.3588 - accuracy: 0.8528 - val_loss: 0.3494 - val_accuracy: 0.8525
Epoch 3/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3453 - accuracy: 0.8587 - val_loss: 0.3541 - val_accuracy: 0.8585
Epoch 4/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3439 - accuracy: 0.8575 - val_loss: 0.3628 - val_accuracy: 0.8585
Epoch 5/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3427 - accuracy: 0.8591 - val_loss: 0.3372 - val_accuracy: 0.8620
Epoch 6/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3397 - accuracy: 0.8630 - val_loss: 0.3434 - val_accuracy: 0.8550
Epoch 7/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3351 - accuracy: 0.8639 - val_loss: 0.3401 - val_accuracy: 0.8610

In [67]:
model.save('churn_model.h5') ##Saving the trained model in HDF5 format for future use

In [4]:
import tensorflow as tf
%load_ext tensorboard
%tensorboard --logdir logs/fit20260317-200440 --port 6008

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6008 (pid 3736), started 0:00:06 ago. (Use '!kill 3736' to kill it.)